In [1]:
# 1.	Биномиальное
# 3 – число признаков
# 3 – число кластеров
# 1- метод к - среднего
# 2 - метод деревьев решений
# 1- эвклидово расстояние

знать структуру и обучение модели нейросети

Z=5 равномерное, P=3 признаков , N=4 классов, взять потом реальные датасеты с задачей кластеризации(?) и сделать на нем
доп задание: на реальном датасете не важно сколько признаков и классов. Примерно 3 лаба мада

стандартизацию делаем после разделения на обучающие и тестовые на обучаемых, стандартизацию делаем независимо от тестовых данных, модель обучается на стандартизированных данных

In [2]:
from sklearn.model_selection import train_test_split

In [3]:
import numpy as np
from sklearn.cluster import KMeans

c1 = np.random.uniform(100, 0.5, (3, 300))
c2 = np.random.uniform(100, 0.5, (3, 300)) + 100
c3 = np.random.uniform(100, 0.5, (3, 300)) + 200
c4 = np.random.uniform(100, 0.5, (3, 300)) + 300

In [4]:
xs = np.concatenate((c1[0], c2[0],c3[0], c4[0]))
ys = np.concatenate((c1[1], c2[1],c3[1], c4[1]))
zs = np.concatenate((c1[2], c2[2],c3[2], c4[2]))



In [5]:
xsyszs = [[xs[i], ys[i],zs[i]] for i in range(len(xs))]
print(len(xsyszs))


1200


разметка кластеров

3 - количество признаков

In [6]:
Y = [[] for i in range(1200)]
for i in range(0, 300):
  Y[i] = 0
for i in range(300, 600):
  Y[i] = 1
for i in range(600, 900):
  Y[i] = 2
for i in range(900, 1200):
  Y[i] = 3

In [7]:
xsyszs=np.array(xsyszs)
xsyszs= xsyszs.reshape((1200,3))

In [8]:
import matplotlib.pyplot as plt
import pandas as pd
import nbformat
import plotly.express as px
#
# fig = plt.figure()
# ax = fig.add_subplot(projection='3d')
# ax.scatter(xsyszs[:,0] , xsyszs[:,1] , xsyszs [:,2], marker='^')

In [9]:
# fig = plt.figure()
# ax = fig.add_subplot(projection='3d')
# ax.scatter(xsyszs[:,0] , xsyszs[:,1] , xsyszs [:,2], marker='o', c=Y)

In [10]:
# import matplotlib.pyplot as plt
# import pandas as pd
# import nbformat
# import plotly.express as px
# res_df = pd.DataFrame(xsyszs)
# fig = px.scatter_3d(res_df, x=0, y=1, z=2)
# fig.show()

In [11]:
res_df = pd.DataFrame(xsyszs)
fig = px.scatter_3d(res_df, x=0, y=1, z=2,color=Y)
fig.show()

cтандартизация

In [12]:
from sklearn.preprocessing import StandardScaler,MinMaxScaler
sc = MinMaxScaler()
xsyszs_n =pd.DataFrame(sc.fit_transform(res_df))
xsyszs_n.describe()


,0,1,2
count,1200.000000,1200.000000,1200.000000
mean,0.496507,0.500783,0.499364
std,0.290223,0.288004,0.289969
min,0.000000,0.000000,0.000000
25%,0.248954,0.251390,0.249409
50%,0.499381,0.501409,0.500490
75%,0.748258,0.750885,0.750552
max,1.000000,1.000000,1.000000


взять последний этот скрин на отчет

In [13]:
fig = px.scatter_3d(xsyszs_n, x=0, y=1, z=2, color=Y)
fig.show()

In [14]:
X_train, X_test, y_train, y_test = train_test_split(xsyszs_n, Y, test_size=0.33, random_state=42)

**Осуществим кластеризацию методом k-средних**

In [15]:
kmeans = KMeans(n_clusters=4, random_state=0).fit(X_train)

In [16]:
df_cluster_centers=pd.DataFrame(kmeans.cluster_centers_)
print(df_cluster_centers)

          0         1         2
0  0.121880  0.131193  0.124481
1  0.875570  0.873883  0.879044
2  0.622364  0.623938  0.626842
3  0.367517  0.374637  0.373261


In [17]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_train,kmeans.labels_))

[[200   0   0   0]
 [  0   0   0 199]
 [  0   0 200   0]
 [  0 205   0   0]]


In [18]:
print(confusion_matrix(y_test,kmeans.predict(X_test)))

[[100   0   0   0]
 [  0   0   0 101]
 [  0   0 100   0]
 [  0  95   0   0]]


In [19]:
import plotly.graph_objects as go
fig = go.Figure(data=[
go.Scatter3d(
    x=X_train[0], y=X_train[1], z=X_train[2], mode='markers', marker=dict(size=2, color=kmeans.labels_)
),

go.Scatter3d(
  x=df_cluster_centers[0],
  y=df_cluster_centers[1],
  z=df_cluster_centers[2],
  mode='markers',
  marker=dict(size=4,color='red'))
]
)
fig.show()

Деление на 2 класса Персептрона с пороговой функцией активации.
Ошибка: смещение должно быть одно, смещение должно быть 1!
https://el.istu.edu/mod/book/view.php?id=218192&chapterid=83497


в оптимальном решение на 1 признак больше! 4
критерий оптимальности, персептрон обучается с учителем. (У учитель) сравниваем ответы нейронки с ответами ошибка 0 для всего обучаеющего набора. целевая функция это ошибка

In [20]:
print(np.unique(y_train, return_counts=True))
from mlxtend.evaluate import accuracy_score

(array([0, 1, 2, 3]), array([200, 199, 200, 205]))


**Персептрон**

In [21]:
class Perceptron:
    def __init__(self, learning_rate=0.01, n_iters=1000, cluster = 0):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.aclivation_func = self._unit_step_func
        self.weights = None
        self.bias = None
        self.cluster = cluster

    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0

        y_ = np.array([1 if (i <=self.cluster) else 0 for i in y])

        print("cluster:", self.cluster)
        print("y_:", np.unique(y_, return_counts=True))

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_predicted = self.aclivation_func(linear_output)

                update = self.lr * (y_[idx] - y_predicted)  #ошибка
                self.weights += update * x_i #новые веса + старые веса = ошибка*на текущий вход р
                self.bias += update # у биас единичный вход = вес единичного входа

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        y_predicted = self.aclivation_func(linear_output)
        # print("linear_output:", np.min(linear_output), np.max(linear_output))
        # print("y_predicted:", np.unique(y_predicted, return_counts=True))
        return y_predicted
#функция активации (если результат сумматора >= 0, то результат = 1)
    def _unit_step_func(self, x):
        return np.where(x>=0, 1, 0)

вес = количество признаков, биас 1

делаем матрицы ошибок на

In [22]:
def art(X, result, net):
    fig = go.Figure(data=[
        go.Scatter3d(
            x=X.iloc[:, 0],
            y=X.iloc[:, 1],
            z=X.iloc[:, 2],
            mode='markers',
            marker=dict(
                size=4,
                color=result
            )
        )
    ])

    fig.update_layout(
        scene=dict(
            xaxis_title='Признак 1',
            yaxis_title='Признак 2',
            zaxis_title='Признак 3'
        )
    )

    x0 = np.linspace(
        np.amin(X.iloc[:, 0]),
        np.amax(X.iloc[:, 0]),
        20
    )
    x1 = np.linspace(
        np.amin(X.iloc[:, 1]),
        np.amax(X.iloc[:, 1]),
        20
    )
    x0, x1 = np.meshgrid(x0, x1)

    # Разделяющая плоскость
    x2 = (
                 -net.weights[0] * x0
                 - net.weights[1] * x1
                 - net.bias
         ) / net.weights[2]

    z_min = np.amin(X.iloc[:, 2])
    z_max = np.amax(X.iloc[:, 2])

    x2_clipped = np.clip(x2, z_min, z_max)

    fig.add_trace(
        go.Surface(
            x=x0,
            y=x1,
            z=x2_clipped,
            opacity=0.8,
        )
    )
    fig.show()

def review(cluster):
    net = Perceptron(cluster = cluster)
    net.fit(X_train.values, y_train)
    result = net.predict(X_train.values)
    print(confusion_matrix(y_train,  result))
    print("weights:", net.weights)
    print("bias:", net.bias)
    art(X_train, result, net)
    print("Тестовая сборка")
    result = net.predict(X_test.values)
    print(confusion_matrix(y_test,  result))
    print("weights:", net.weights)
    print("bias:", net.bias)
    art(X_test, result, net)



In [23]:
review(0)

cluster: 0
y_: (array([0, 1]), array([604, 200]))
[[  0 200   0   0]
 [199   0   0   0]
 [200   0   0   0]
 [205   0   0   0]]
weights: [-0.01510868 -0.01183023 -0.01340649]
bias: 0.01


Тестовая сборка
[[  0 100   0   0]
 [101   0   0   0]
 [100   0   0   0]
 [ 95   0   0   0]]
weights: [-0.01510868 -0.01183023 -0.01340649]
bias: 0.01


In [24]:
review(1)


cluster: 1
y_: (array([0, 1]), array([405, 399]))
[[  0 200   0   0]
 [  0 199   0   0]
 [200   0   0   0]
 [205   0   0   0]]
weights: [-0.0143856  -0.01090282 -0.01651221]
bias: 0.02


Тестовая сборка
[[  0 100   0   0]
 [  0 101   0   0]
 [100   0   0   0]
 [ 95   0   0   0]]
weights: [-0.0143856  -0.01090282 -0.01651221]
bias: 0.02


In [25]:
review(2)

cluster: 2
y_: (array([0, 1]), array([205, 599]))
[[  0 200   0   0]
 [  0 199   0   0]
 [  0 200   0   0]
 [205   0   0   0]]
weights: [-0.02104928 -0.02286499 -0.02211087]
bias: 0.05


Тестовая сборка
[[  0 100   0   0]
 [  0 101   0   0]
 [  0 100   0   0]
 [ 95   0   0   0]]
weights: [-0.02104928 -0.02286499 -0.02211087]
bias: 0.05


круглов нечеткая логика и нейронные сети сеть кохонена

сеть кохонена только на реальном

предиктим на тестовой и обучающей, плоскость на обучающей, на тестовой просто выводим как разметил

вывести плоскость разделения персептрона, где полоса будет делить облако 0 и остальные как 1

**Сеть Кохонена**

In [26]:
class KohonenNet():
    def __init__(self, m=3, n=3, lr=1, sigma=1, max_iter=3000, weights = []):
        self.m = m #количество классов
        self.n = n #количество уровней сети
        self.shape = (m, n)
        self.initial_lr = lr
        self.lr = lr #скорость обучения
        self.sigma = sigma #параметр изменения скорости обучения (она будет снижаться с каждой итерацией)
        self.max_iter = max_iter

        self.weights = weights

#функция нахождения выйгравшего нейрона (нейрона с минимальным расстоянием до точки)
    def _find_bmu(self, x):
        x_stack = np.stack([x]*(self.m*self.n), axis=0)#матрица размером [количество центров кластеров(нейронов), количество признаков] (для каждого веса своя строка признаков(точка))
        distance = np.linalg.norm(x_stack - self.weights, axis=1)#эвклидовы расстояния между точкой выборки и нейронами
        return np.argmin(distance)#возвращаем индекс минимального расстояния (индекс подходящего нейрона)

    def step(self, x):
        x_stack = np.stack([x]*(self.m*self.n), axis=0)#матрица размером [количество весов, количество признаков] (для каждого веса своя строка признаков(точка))

        bmu_index = self._find_bmu(x)#передаем в функцию строку признаков (точку) и получаем индекс выйгравшего нейрона (центра кластера)
        self.weights[bmu_index] += self.lr * (x - self.weights[bmu_index])

    #обучение весов
    def fit(self, X, epochs=1, shuffle=True):
        global_iter_counter = 0
        n_samples = X.shape[0]#количество элементов выборки
        total_iterations = np.minimum(epochs * n_samples, self.max_iter)#количество обучения весов в эпохе

        for epoch in range(epochs):#изначально у нас 1 эпоха
            if global_iter_counter > self.max_iter:#не даем проводить больше 3000 эпох
                break

            if shuffle:
                indices = np.random.permutation(n_samples)#создание массива с индексами в разброс
            else:
                indices = np.arange(n_samples)#создание массива с индексами по порядку

            #обучение
            for idx in indices:#проход по выборке с индексами idx
                if global_iter_counter > self.max_iter:
                    break
                input = X[idx]
                #перемещение нейрона(изменение весов победившего нейрона)
                self.step(input)

                global_iter_counter += 1
                self.lr = (1 - (global_iter_counter / total_iterations)) * self.initial_lr#изменение параметра обучения

        self._n_iter_ = global_iter_counter

        return

    def returnChangedWeights(self):
      return self.weights

    def predict(self, X):
        labels = np.array([self._find_bmu(x) for x in X])
        return labels


Алгоритм, ответственный за формирование сети Кохонена, начинается с инициализации синоптических весов сети. Обычно это происходит с помощью назначения синоптическим весам малых значений, сформированных генератором случайных чисел. После корректной инициализации сети для формирования сети запускаются три следующих основных процесса.

Существуют различные механизмы учета активности нейронов в
процессе обучения. Часто используется метод подсчета потенциала pi каждого
нейрона, значение которого модифицируется всякий раз после предъявления
очередной реализации входного вектора х в соответствии со следующей
формулой (в ней предполагается, что победителем стал w-й нейрон):

https://microtechnics.ru/nejronnaya-set-kohonena-samoorganizuyushhiesya-karty-obuchenie/

In [27]:
weights = np.random.uniform(
    0.4,
    0.6,
    size=(4, 3)
)


In [28]:
net = KohonenNet(m=4, n=1, weights=weights)
net.fit(X_train.values, shuffle=True)

result_train = net.predict(X_train.values)

result_test = net.predict(X_test.values)

In [29]:
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X_test.iloc[:, 0],
        y=X_test.iloc[:, 1],
        z=X_test.iloc[:, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=result_test
        ),
        name='Тестовые объекты'
    )
)

changed_weights = net.returnChangedWeights()

fig.add_trace(
    go.Scatter3d(
        x=changed_weights[:, 0],
        y=changed_weights[:, 1],
        z=changed_weights[:, 2],
        mode='markers',
        marker=dict(
            size=8,
            color='red',
            symbol='diamond'
        ),
        name='Нейроны'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='Признак 1',
        yaxis_title='Признак 2',
        zaxis_title='Признак 3'
    ),
    title='Результат кластеризации тестовой выборки сетью Кохонена'
)

fig.show()

In [30]:
from sklearn.metrics import classification_report

print("Матрица ошибок для обучающей выборки:")
print(confusion_matrix(y_train, result_train))

print("\nClassification Report для обучающей выборки:")
print(classification_report(y_train, result_train))

print("Матрица ошибок для тестовой выборки:")
print(confusion_matrix(y_test, result_test))

print("\nClassification Report для тестовой выборки:")
print(classification_report(y_test, result_test))

Матрица ошибок для обучающей выборки:
[[200   0   0   0]
 [  0   0   0 199]
 [  0   0 200   0]
 [  0 205   0   0]]

Classification Report для обучающей выборки:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       0.00      0.00      0.00       199
           2       1.00      1.00      1.00       200
           3       0.00      0.00      0.00       205

    accuracy                           0.50       804
   macro avg       0.50      0.50      0.50       804
weighted avg       0.50      0.50      0.50       804

Матрица ошибок для тестовой выборки:
[[100   0   0   0]
 [  0   0   0 101]
 [  0   0 100   0]
 [  0  95   0   0]]

Classification Report для тестовой выборки:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       0.00      0.00      0.00       101
           2       1.00      1.00      1.00       100
           3       0.00     

In [31]:
print("Число итераций обучения:", net._n_iter_)

Число итераций обучения: 804


**Вероятностная нейронная сеть PNN**

In [32]:
import math

# Probabilistic Neural Network with 4 layers
class PNN(object):
    def __init__(self):
        self.L2 = []    # Layer 2 that holds the patterns
        print('Empty PNN created.')

    def train(self, X, y, p=4):
        self.n_ = X.shape[1]  # num of features
        self.p_ = p           # num of classes

        # Layer 2 (Pattern): Set up empty lists for each class

        for k in range(self.p_):
            self.L2.append([])    # Using Python's basic lists because ndarray cannot append empty arrays
                                  # Also perhaps we might have to input different data types

        # Enter patterns into Layer 2
        for i in range(X.shape[0]):
            self.L2[y[i]].append(X[i])


        self.L2 = np.array(self.L2, dtype=object)    # Change to ndarray for speed (Is this faster?)

        print('PNN with %d classes trained.' % self.p_)

    def crossValidate(self, X, y, sigma=0.5):
        result = self.predict(X, sigma)
        num_correct = sum(result[:, 0] == y)

        print('Cross validation accuracy with sigma %.2f: %.1f%%' % (sigma, num_correct/len(y) * 100))

    def predict(self, X, sigma=0.5):
        m = X.shape[0]
        accL3 = np.zeros((m, self.p_))
        accL4 = np.zeros(m)

        self.sigma_ = sigma    # smoothing parameter, not standard deviation
        self.C1_ = 2 * self.sigma_**2
        C2_ = (math.sqrt(2*math.pi) * self.sigma_) ** (- self.n_)

        # Layer 1 (Input): x
        for i in range(m):
            x = X[i]

            # Layer 3 (Averaging): for each class
            self.L3_ = np.zeros(self.p_)
            for k in range(self.p_):
                for ki in range(len(self.L2[k])):
                    self.L3_[k] += self._activation(x, self.L2[k][ki])
                self.L3_[k] /= len(self.L2[k])

                # Multiply constant
                self.L3_[k] *= C2_
                accL3[i][k] = self.L3_[k]

            # Layer 4 (Output/Decision): Maxing
            self.L4_ = self.L3_.argmax()
            accL4[i] = self.L4_

        return np.column_stack((accL4, accL3))

    def _activation(self, x, w):
        diff = x - w
        return math.exp( - np.dot(diff, diff) / self.C1_ )


# Normalize to unit length: [0, 1]
# X must be ndarray
def Normalize(X):
    x_max = X.max(axis=0)
    x_min = X.min(axis=0)
    return (X - x_min) / (x_max - x_min)

In [33]:
#from neupy import algorithms

X_train_pnn = np.asarray(X_train)
y_train_pnn = np.asarray(y_train)

X_test_pnn = np.asarray(X_test)
y_test_pnn = np.asarray(y_test)

pnn = PNN()

pnn.train(
    X_train_pnn,
    y_train_pnn,
    p=4
)

prY_Train = pnn.predict(
    X_train_pnn,
    sigma=0.5
)

prY_Test = pnn.predict(
    X_test_pnn,
    sigma=0.5
)



Empty PNN created.
PNN with 4 classes trained.


In [34]:
pred_train = prY_Train[:, 0].astype(int)
pred_test = prY_Test[:, 0].astype(int)

print("Матрица ошибок для обучающей выборки:")
print(confusion_matrix(y_train, pred_train))

print("\nClassification Report для обучающей выборки:")
print(classification_report(y_train, pred_train))

print("Матрица ошибок для тестовой выборки:")
print(confusion_matrix(y_test, pred_test))

print("\nClassification Report для тестовой выборки:")
print(classification_report(y_test, pred_test))


Матрица ошибок для обучающей выборки:
[[200   0   0   0]
 [  0 199   0   0]
 [  0   0 200   0]
 [  0   0   0 205]]

Classification Report для обучающей выборки:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       199
           2       1.00      1.00      1.00       200
           3       1.00      1.00      1.00       205

    accuracy                           1.00       804
   macro avg       1.00      1.00      1.00       804
weighted avg       1.00      1.00      1.00       804

Матрица ошибок для тестовой выборки:
[[100   0   0   0]
 [  0 101   0   0]
 [  0   0 100   0]
 [  0   0   0  95]]

Classification Report для тестовой выборки:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       101
           2       1.00      1.00      1.00       100
           3       1.00     

In [35]:
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X_test_pnn[:, 0],
        y=X_test_pnn[:, 1],
        z=X_test_pnn[:, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=pred_test
        ),
        name='Тестовые объекты'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='Признак 1',
        yaxis_title='Признак 2',
        zaxis_title='Признак 3'
    ),
    title='Результат классификации тестовой выборки вероятностной сетью'
)

fig.show()

**MLP**

In [36]:
# Backprop on the Seeds Dataset
from random import seed
from random import randrange
from random import random
from csv import reader
from math import exp

# Load a CSV file
def load_csv(filename):
	dataset = list()
	with open(filename, 'r') as file:
		csv_reader = reader(file)
		for row in csv_reader:
			if not row:
				continue
			dataset.append(row)
	return dataset

# Convert string column to float
def str_column_to_float(dataset, column):
	for row in dataset:
		row[column] = float(row[column].strip())

# Convert string column to integer
def str_column_to_int(dataset, column):
	class_values = [row[column] for row in dataset]
	unique = set(class_values)
	lookup = dict()
	for i, value in enumerate(unique):
		lookup[value] = i
	for row in dataset:
		row[column] = lookup[row[column]]
	return lookup

# Find the min and max values for each column
def dataset_minmax(dataset):
	minmax = list()
	stats = [[min(column), max(column)] for column in zip(*dataset)]
	return stats

# Rescale dataset columns to the range 0-1
def normalize_dataset(dataset, minmax):
	for row in dataset:
		for i in range(len(row)-1):
			row[i] = (row[i] - minmax[i][0]) / (minmax[i][1] - minmax[i][0])

# Split a dataset into k folds
def cross_validation_split(dataset, n_folds):
	dataset_split = list()
	dataset_copy = list(dataset)
	fold_size = int(len(dataset) / n_folds)
	for i in range(n_folds):
		fold = list()
		while len(fold) < fold_size:
			index = randrange(len(dataset_copy))
			fold.append(dataset_copy.pop(index))
		dataset_split.append(fold)
	return dataset_split

# Calculate accuracy percentage
def accuracy_metric(actual, predicted):
	correct = 0
	for i in range(len(actual)):
		if actual[i] == predicted[i]:
			correct += 1
	return correct / float(len(actual)) * 100.0

# Evaluate an algorithm using a cross validation split
def evaluate_algorithm(dataset, algorithm, n_folds, *args):
	folds = cross_validation_split(dataset, n_folds)
	scores = list()
	for fold in folds:
		train_set = list(folds)
		train_set.remove(fold)
		train_set = sum(train_set, [])
		test_set = list()
		for row in fold:
			row_copy = list(row)
			test_set.append(row_copy)
			row_copy[-1] = None
		predicted = algorithm(train_set, test_set, *args)
		actual = [row[-1] for row in fold]
		accuracy = accuracy_metric(actual, predicted)
		scores.append(accuracy)
	return scores

# Calculate neuron activation for an input
def activate(weights, inputs):
	activation = weights[-1]
	for i in range(len(weights)-1):
		activation += weights[i] * inputs[i]
	return activation

# Transfer neuron activation
def transfer(activation):
	return 1.0 / (1.0 + exp(-activation))

# Forward propagate input to a network output
def forward_propagate(network, row):
	inputs = row
	for layer in network:
		new_inputs = []
		for neuron in layer:
			activation = activate(neuron['weights'], inputs)
			neuron['output'] = transfer(activation)
			new_inputs.append(neuron['output'])
		inputs = new_inputs
	return inputs

# Calculate the derivative of an neuron output
def transfer_derivative(output):
	return output * (1.0 - output)

# Backpropagate error and store in neurons
def backward_propagate_error(network, expected):
	for i in reversed(range(len(network))):
		layer = network[i]
		errors = list()
		if i != len(network)-1:
			for j in range(len(layer)):
				error = 0.0
				for neuron in network[i + 1]:
					error += (neuron['weights'][j] * neuron['delta'])
				errors.append(error)
		else:
			for j in range(len(layer)):
				neuron = layer[j]
				errors.append(neuron['output'] - expected[j])
		for j in range(len(layer)):
			neuron = layer[j]
			neuron['delta'] = errors[j] * transfer_derivative(neuron['output'])

# Update network weights with error
def update_weights(network, row, l_rate):
	for i in range(len(network)):
		inputs = row[:-1]
		if i != 0:
			inputs = [neuron['output'] for neuron in network[i - 1]]
		for neuron in network[i]:
			for j in range(len(inputs)):
				neuron['weights'][j] -= l_rate * neuron['delta'] * inputs[j]
			neuron['weights'][-1] -= l_rate * neuron['delta']

# Train a network for a fixed number of epochs
def train_network(network, train, l_rate, n_epoch, n_outputs):
	for epoch in range(n_epoch):
		for row in train:
			outputs = forward_propagate(network, row)
			expected = [0 for i in range(n_outputs)]
			expected[row[-1]] = 1
			backward_propagate_error(network, expected)
			update_weights(network, row, l_rate)

# Initialize a network
def initialize_network(n_inputs, n_hidden, n_outputs):
	network = list()
	hidden_layer = [{'weights':[random() for i in range(n_inputs + 1)]} for i in range(n_hidden)]
	network.append(hidden_layer)
	output_layer = [{'weights':[random() for i in range(n_hidden + 1)]} for i in range(n_outputs)]
	network.append(output_layer)
	return network

# Make a prediction with a network
def predict(network, row):
	outputs = forward_propagate(network, row)
	return outputs.index(max(outputs))

# Backpropagation Algorithm With Stochastic Gradient Descent
def back_propagation(train, test, l_rate, n_epoch, n_hidden):
	n_inputs = len(train[0]) - 1
	n_outputs = len(set([row[-1] for row in train]))
	network = initialize_network(n_inputs, n_hidden, n_outputs)
	train_network(network, train, l_rate, n_epoch, n_outputs)
	predictions = list()
	for row in test:
		prediction = predict(network, row)
		predictions.append(prediction)
	return(predictions)


In [37]:

X_train_mlp = np.asarray(X_train, dtype=float)
X_test_mlp = np.asarray(X_test, dtype=float)

y_train_mlp = np.asarray(y_train, dtype=int)
y_test_mlp = np.asarray(y_test, dtype=int)

print("Классы:", np.unique(
    y_train_mlp,
    return_counts=True
))

n_inputs = X_train_mlp.shape[1]
n_outputs = len(np.unique(y_train_mlp))

print("Количество входных признаков:", n_inputs)
print("Количество классов:", n_outputs)

train_dataset = [
    list(x) + [int(y)]
    for x, y in zip(
        X_train_mlp,
        y_train_mlp
    )
]

seed(42)

n_hidden = 2
l_rate = 0.5
n_epoch = 200

network = initialize_network(
    n_inputs,
    n_hidden,
    n_outputs
)

train_network(
    network,
    train_dataset,
    l_rate,
    n_epoch,
    n_outputs
)

n_iterations = n_epoch * len(train_dataset)

print("Число итераций обучения:", n_iterations)

Классы: (array([0, 1, 2, 3]), array([200, 199, 200, 205]))
Количество входных признаков: 3
Количество классов: 4
Число итераций обучения: 160800


In [38]:

pred_train = np.array([
    predict(network, row)
    for row in train_dataset
])

pred_test = np.array([
    predict(network, row)
    for row in X_test_mlp.tolist()
])

In [39]:
print("Матрица ошибок — Train:")
print(confusion_matrix(
    y_train_mlp,
    pred_train
))

print("\nClassification Report — Train:")
print(classification_report(
    y_train_mlp,
    pred_train
))

print("\nМатрица ошибок — Test:")
print(confusion_matrix(
    y_test_mlp,
    pred_test
))

print("\nClassification Report — Test:")
print(classification_report(
    y_test_mlp,
    pred_test
))

Матрица ошибок — Train:
[[200   0   0   0]
 [  0 199   0   0]
 [  0   0 200   0]
 [  0   0   0 205]]

Classification Report — Train:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       199
           2       1.00      1.00      1.00       200
           3       1.00      1.00      1.00       205

    accuracy                           1.00       804
   macro avg       1.00      1.00      1.00       804
weighted avg       1.00      1.00      1.00       804


Матрица ошибок — Test:
[[100   0   0   0]
 [  0 101   0   0]
 [  0   0 100   0]
 [  0   0   0  95]]

Classification Report — Test:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       101
           2       1.00      1.00      1.00       100
           3       1.00      1.00      1.00        95

    accuracy                

In [40]:
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X_test_mlp[:, 0],
        y=X_test_mlp[:, 1],
        z=X_test_mlp[:, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=pred_test
        ),
        name='Тестовые объекты'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='Признак 1',
        yaxis_title='Признак 2',
        zaxis_title='Признак 3'
    ),
    title='Результат классификации тестовой выборки сетью MLP'
)

fig.show()

**MLP средствами TensorFlow**

In [41]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)

X_train_tf = np.asarray(X_train, dtype=np.float32)
X_test_tf = np.asarray(X_test, dtype=np.float32)
y_train_tf = np.asarray(y_train, dtype=np.int32)
y_test_tf = np.asarray(y_test, dtype=np.int32)

model_tf = Sequential([
    Input(shape=(3,)),
    Dense(16, activation='relu'),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dropout(0.3),
    Dense(4, activation='softmax')
])

model_tf.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_tf.summary()

early_stopping_tf = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

history_tf = model_tf.fit(
    X_train_tf,
    y_train_tf,
    epochs=600,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping_tf],
    verbose=0
)

print("Количество эпох обучения:", len(history_tf.history['loss']))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            36 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 236 (944.00 B)

 Trainable params: 236 (944.00 B)

 Non-trainable params: 0 (0.00 B)

Количество эпох обучения: 262


In [42]:
pred_train_tf = np.argmax(
    model_tf.predict(X_train, verbose=0),
    axis=1
)

pred_test_tf = np.argmax(
    model_tf.predict(X_test, verbose=0),
    axis=1
)

print("Матрица ошибок — TensorFlow MLP — Train:")
print(confusion_matrix(y_train, pred_train_tf))

print("\nClassification Report — TensorFlow MLP — Train:")
print(classification_report(y_train, pred_train_tf))

print("\nМатрица ошибок — TensorFlow MLP — Test:")
print(confusion_matrix(y_test, pred_test_tf))

print("\nClassification Report — TensorFlow MLP — Test:")
print(classification_report(y_test, pred_test_tf))

Матрица ошибок — TensorFlow MLP — Train:
[[200   0   0   0]
 [  0 199   0   0]
 [  0   0 200   0]
 [  0   0   1 204]]

Classification Report — TensorFlow MLP — Train:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       199
           2       1.00      1.00      1.00       200
           3       1.00      1.00      1.00       205

    accuracy                           1.00       804
   macro avg       1.00      1.00      1.00       804
weighted avg       1.00      1.00      1.00       804


Матрица ошибок — TensorFlow MLP — Test:
[[100   0   0   0]
 [  0 101   0   0]
 [  0   0 100   0]
 [  0   0   0  95]]

Classification Report — TensorFlow MLP — Test:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       101
           2       1.00      1.00      1.00       100
           3   

In [43]:
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X_test.iloc[:, 0],
        y=X_test.iloc[:, 1],
        z=X_test.iloc[:, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=pred_test_tf
        ),
        name='Тестовые объекты'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='Признак 1',
        yaxis_title='Признак 2',
        zaxis_title='Признак 3'
    ),
    title='Результат классификации тестовой выборки сетью MLP TensorFlow'
)

fig.show()

In [44]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=history_tf.history['loss'],
        mode='lines',
        name='Обучающая выборка'
    )
)

fig.add_trace(
    go.Scatter(
        y=history_tf.history['val_loss'],
        mode='lines',
        name='Валидационная выборка'
    )
)

fig.update_layout(
    title='Изменение функции ошибки при обучении MLP TensorFlow',
    xaxis_title='Эпоха',
    yaxis_title='Loss'
)

fig.show()

**pytorch**

In [45]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
np.random.seed(42)

In [46]:
class MLP_PyTorch(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(8, 4)
        )

    def forward(self, x):
        return self.network(x)

def predict_torch(model, X):

    model.eval()

    with torch.no_grad():
        output = model(X)
        prediction = torch.argmax(output, dim=1)

    return prediction.numpy()

In [47]:
X_train_torch = torch.tensor(
    X_train.values,
    dtype=torch.float32
)

X_test_torch = torch.tensor(
    X_test.values,
    dtype=torch.float32
)

y_train_torch = torch.tensor(
    np.asarray(y_train),
    dtype=torch.long
)

y_test_torch = torch.tensor(
    np.asarray(y_test),
    dtype=torch.long
)

model_torch = MLP_PyTorch()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model_torch.parameters(),
    lr=0.001
)

X_train_torch_part, X_val_torch, y_train_torch_part, y_val_torch = train_test_split(
    X_train_torch,
    y_train_torch,
    test_size=0.2,
    random_state=42,
    stratify=y_train_torch
)

In [48]:
max_epochs = 600
patience = 20

best_val_loss = float('inf')
best_model = None
epochs_without_improvement = 0

train_losses = []
val_losses = []

for epoch in range(max_epochs):

    model_torch.train()

    optimizer.zero_grad()

    output = model_torch(X_train_torch_part)
    loss = criterion(output, y_train_torch_part)

    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    model_torch.eval()

    with torch.no_grad():
        val_output = model_torch(X_val_torch)
        val_loss = criterion(val_output, y_val_torch)

    val_losses.append(val_loss.item())

    # Early Stopping
    if val_loss.item() < best_val_loss:

        best_val_loss = val_loss.item()

        best_model = {
            key: value.clone()
            for key, value in model_torch.state_dict().items()
        }

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping на эпохе:", epoch + 1)
        break

model_torch.load_state_dict(best_model)

print("Количество эпох обучения:", len(train_losses))

Количество эпох обучения: 600


In [49]:
X_train_torch_full = torch.tensor(
    X_train.values,
    dtype=torch.float32
)

y_train_torch_full = torch.tensor(
    np.asarray(y_train),
    dtype=torch.long
)

pred_train_torch = predict_torch(
    model_torch,
    X_train_torch
)

pred_test_torch = predict_torch(
    model_torch,
    X_test_torch
)

print("Матрица ошибок — PyTorch MLP — Train:")
print(confusion_matrix(y_train, pred_train_torch))

print("\nClassification Report — PyTorch MLP — Train:")
print(classification_report(y_train, pred_train_torch))

print("\nМатрица ошибок — PyTorch MLP — Test:")
print(confusion_matrix(y_test, pred_test_torch))

print("\nClassification Report — PyTorch MLP — Test:")
print(classification_report(y_test, pred_test_torch))

Матрица ошибок — PyTorch MLP — Train:
[[200   0   0   0]
 [  0 199   0   0]
 [  0   0 199   1]
 [  0   0   0 205]]

Classification Report — PyTorch MLP — Train:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       199
           2       1.00      0.99      1.00       200
           3       1.00      1.00      1.00       205

    accuracy                           1.00       804
   macro avg       1.00      1.00      1.00       804
weighted avg       1.00      1.00      1.00       804


Матрица ошибок — PyTorch MLP — Test:
[[100   0   0   0]
 [  0 101   0   0]
 [  0   0 100   0]
 [  0   0   0  95]]

Classification Report — PyTorch MLP — Test:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       101
           2       1.00      1.00      1.00       100
           3       1.00    

In [50]:
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X_test.iloc[:, 0],
        y=X_test.iloc[:, 1],
        z=X_test.iloc[:, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=pred_test_torch
        ),
        name='Тестовые объекты'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='Признак 1',
        yaxis_title='Признак 2',
        zaxis_title='Признак 3'
    ),
    title='Результат классификации тестовой выборки сетью MLP PyTorch'
)

fig.show()

In [51]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=train_losses,
        mode='lines',
        name='Обучающая выборка'
    )
)

fig.add_trace(
    go.Scatter(
        y=val_losses,
        mode='lines',
        name='Валидационная выборка'
    )
)

fig.update_layout(
    title='Изменение функции ошибки при обучении MLP PyTorch',
    xaxis_title='Эпоха',
    yaxis_title='Loss'
)

fig.show()